# Two-stage logical reconstruction (the schema-inference pipeline)

**Company server, PRIVATE data.** The approach `layout_description.md` calls for:
infer the logical schema, then assign every fragment to its cell — not conventional
TSR. Decoupled into two local stages:

1. **Stage 1** — OCR + geometry: PaddleOCR words → a grid/coords block, and
   (optionally) a MinerU2.5-Pro table-structure draft as the specialist ceiling.
2. **Stage 2** — a schema-conditioned reasoning VLM (thinking on) infers the logical
   schema from stage-1 evidence — **document-agnostic, no hard-coded layout** (a known
   header is passed only as an optional hint to verify) — emits logical HTML, keeps
   blanks blank, and tags each non-empty cell with a `data-bbox` so the cells can be drawn.

Input images are already **cropped to the table by a detector (YOLOX)**, so the prompt
also guards against a clipped outer border being read as a dropped/merged column.

## 1. Config

In [ ]:
import sys; sys.path.insert(0, "..")
from pathlib import Path

from src.model.registry import MODEL_QWEN36_35B_FP8
from src.model.vllm_client import VLLMTableReconstructor
from src.model.prompts import build_schema_instruction, DEFAULT_INVOICE_SCHEMA
from src.ocr.engine import run_ocr
from src.ocr.layout import serialize_layout

IMAGES_DIR = Path("data/invoices")
# THE one instruction — identical in teacher-label-tables.ipynb and
# finetune-and-serve.ipynb. Document-agnostic by default: infer the schema from the
# body, do NOT pin one layout (a prompt pinned to one sample only reconstructs
# documents that match it).
# with_bbox=True -> the model tags every non-empty cell with data-bbox so we can
# draw them; it must be True on EVERY call or boxes appear on one call and not the
# next. Pass a known header only as an optional hint to verify, e.g.:
#     build_schema_instruction(DEFAULT_INVOICE_SCHEMA, with_bbox=True)
INSTRUCTION = build_schema_instruction(with_bbox=True)
reasoner = VLLMTableReconstructor(model_id=MODEL_QWEN36_35B_FP8, base_url="http://localhost:8000/v1", thinking=True)
img = sorted(IMAGES_DIR.glob("*.png"))[0]
print("image:", img)

## 2. Stage 1a — OCR + geometry grounding

In [ ]:
layout = serialize_layout(run_ocr(img), style="grid")
print(layout[:1200])

## 3. Stage 1b (optional) — MinerU2.5-Pro structure draft (specialist ceiling)

A pure parser recovers the *visible* structure; it will miss implicit/optional
columns, which is exactly why stage 2 exists. Useful as a comparison floor.

In [ ]:
from src.model.mineru_client import MinerUTableReconstructor
mineru = MinerUTableReconstructor()
draft = mineru.predict(img)
print(draft.html[:1500] if draft.html else "(no table extracted — confirm MinerU output shape)")

## 4. Stage 2 — schema-conditioned reasoning → logical HTML

In [ ]:
# guided_regex forces output to a single <table>...</table> (data-bbox attrs allowed) and blocks stray prose.
pred = reasoner.predict(
    img, instruction=INSTRUCTION, ocr_layout=layout,
    guided_regex=r"<table>[\s\S]*</table>",
)
from IPython.display import HTML, Image as IPyImage, display
from src.demo.boxes import draw_cell_boxes, boxed_cells

display(HTML(pred.html or "<i>empty</i>"))

# Draw the per-cell boxes the model emitted (with_bbox) onto the cropped table:
# <th> header cells in blue, body cells in red; blank cells carry no box.
overlay = draw_cell_boxes(img, pred.html)
out = img.with_name(img.stem + "_boxes.png")
overlay.save(out)
print(f"drew {len(boxed_cells(pred.html))} cell boxes -> {out}")
display(overlay)

## 4b. Alternative output — JSON cell list (guided; boxes structurally mandatory)

The output format is free, so the recommended high-accuracy path emits a flat JSON
cell list instead of HTML: `guided_json` makes a bbox+page **required on every
cell** (dropping boxes becomes impossible, not just off-prompt), blank cells are
grid holes the model cannot shift a value into, and `columns` must be committed
before any cell (schema-first reasoning enforced by field order).
`validate_cells` machine-checks the answer — overlaps, out-of-range columns, and
(given the stage-1 words) invented/dropped values — and feeds one repair
round-trip. `cells_to_html` converts losslessly for the existing metrics/renderer.

In [ ]:
from src.model.cells import (
    build_cells_instruction, cells_json_schema, parse_cells,
    validate_cells, build_repair_suffix, cells_to_html,
)

CELLS_INS = build_cells_instruction()          # same optional-hint rule as build_schema_instruction
words = run_ocr(img)                            # stage-1 words, reused for value-coverage checks

cpred = reasoner.predict(img, instruction=CELLS_INS, ocr_layout=layout,
                         guided_json=cells_json_schema())
payload = parse_cells(cpred.raw)
problems = validate_cells(payload, n_pages=1, words=words)
print(f"{len(payload['cells']) if payload else 0} cells, {len(problems)} violations")
for p in problems[:10]:
    print(" -", p)

if problems:  # one repair round-trip — still a fresh single-turn request
    cpred = reasoner.predict(
        img, instruction=CELLS_INS + "\n\n" + build_repair_suffix(cpred.raw, problems),
        ocr_layout=layout, guided_json=cells_json_schema(),
    )
    payload = parse_cells(cpred.raw)
    print("after repair:", len(validate_cells(payload, n_pages=1, words=words)), "violations")

cells_html = cells_to_html(payload)             # -> existing metrics / renderer, unchanged
display(HTML(cells_html or "<i>empty</i>"))
display(draw_cell_boxes(img, cells_html))

## 4c. Long tables split across pages — pass a list of images

A table that continues onto a second crop goes in as `[page1, page2]` (reading
order). The prompt gains the stitching note automatically: one logical table, a
repeated continuation header emitted once, a row cut at the page break merged, and
every position tagged `data-page="N"` (HTML path) / `"p": N` (cells path) in that
image's own pixel space. Works identically on both output paths; pass
`ocr_layout=[layout1, layout2]` for per-page grounding.

In [ ]:
pages = sorted(IMAGES_DIR.glob("*.png"))[:2]   # point at the crops of ONE long table
if len(pages) > 1:
    page_layouts = [serialize_layout(run_ocr(p), style="grid") for p in pages]

    # cells path (recommended): p is required on every cell by the guided schema
    mp = reasoner.predict(pages, instruction=CELLS_INS, ocr_layout=page_layouts,
                          guided_json=cells_json_schema())
    mp_payload = parse_cells(mp.raw)
    print(len(validate_cells(mp_payload, n_pages=len(pages), words=[w for p in pages for w in run_ocr(p)])),
          "violations")
    display(HTML(cells_to_html(mp_payload) or "<i>empty</i>"))

    # HTML path: same call shape, data-page appears on every non-empty cell
    mp_html = reasoner.predict(pages, instruction=INSTRUCTION, ocr_layout=page_layouts,
                               guided_regex=r"<table>[\s\S]*</table>")
    display(HTML(mp_html.html or "<i>empty</i>"))
else:
    print("only one image found — put both page crops of a long table in", IMAGES_DIR)

## 5. Compare the two stages

If you have any corrected label for this image, score both with the schema metrics
(`content_placement`, `blank_preservation`, `schema_match`) to see the reasoning
pass recover the implicit columns the specialist dropped.

In [ ]:
true_html = None   # paste a corrected label here to score
if true_html:
    from src.eval.metrics import content_placement, blank_preservation, schema_match
    for name, html in [("mineru", draft.html), ("stage-2", pred.html)]:
        cp, bp, sm = content_placement(html, true_html), blank_preservation(html, true_html), schema_match(html, true_html)
        print(f"{name:<8} placement={cp.accuracy:.2f} blank_keep={bp.rate:.2f} cols={sm.pred_cols}/{sm.true_cols} ok={sm.cols_correct}")

---

## 6. Full Qwen-only staged pipeline (recommended end-to-end)

`src/pipeline.py` assembles the whole `pipeline_design.md` Architecture-A path into
one object. **Qwen reads its own OCR** (tiled, structure-blind), a deterministic
survey measures the geometry, Qwen abduces a per-document schema that a trial scorer
*tests* (not trusts), cells are emitted as **fragment-id references** (invention
impossible, drops computable), `k` drafts are voted for per-cell confidence, and the
layout-independent verifier drives a **bounded, monotone** repair. No hard-coded
layout; the client hits loopback, nothing leaves the box. `from_client` wires it to
the served teacher in one line, and the `run_pipeline.py` CLI runs this same object
headless.

Set `use_schema_hint=False` (CLI: `--no-schema`) and compare to the default — that
is the **B-vs-A go/no-go ablation** (`pipeline_design.md` §8). **First run is a
debugging session:** start on one image and sanity-check the fragment count, the
logical-column count, and the problem list before trusting the HTML.

This is the reconstruction path the fine-tuned student is distilled to reproduce.

In [ ]:
from src.pipeline import StagedPipeline
from IPython.display import HTML, display
from src.demo.boxes import draw_cell_boxes

# One line wires the reading pass + schema discovery + grounded cells to the served
# teacher. k>1 votes for per-cell confidence; repair is bounded and monotone.
# use_schema_hint=False is the Architecture-B baseline for the B-vs-A go/no-go.
pipe = StagedPipeline.from_client(reasoner, k=1, max_repair_rounds=1)
result = pipe.run(img)                      # a path, or [page1, page2] for ONE long table

cols = result.payload.get("columns", [])
print(f"{len(result.fragments)} fragments -> {len(result.payload.get('cells', []))} cells, "
      f"{len(cols)} logical columns: " + ", ".join(c.get("name", "?") for c in cols[:8]))
print("triage confidence:", result.table_confidence, "(uncalibrated)")
if result.schema_hint:
    print("schema hint:\n  " + result.schema_hint.replace("\n", "\n  "))
for n in result.notes:               print(" .", n)
for p in result.report.problems[:8]: print(" problem:", p)
for f in result.report.flags[:8]:    print(" flag:", f)

display(HTML(result.html or "<i>empty</i>"))
display(draw_cell_boxes(img, result.html))   # per-cell boxes: <th> blue, body red